In [1]:
import os
import sys
sys.path.append('/lustre/hawcz01/scratch/userspace/rbabu/J1809-Pass5.1/gamera/gam/GAMERA/lib') 
sys.path.append('rad_field_richard_tuffs')
import gappa as gp
import numpy as np
import astropy.constants as c
import astropy.units as u
import emcee
from multiprocessing import Pool
from scipy.io import readsav
from astropy.table import Table
from clean_model import *
import matplotlib.pyplot as plt

In [2]:
fu = gp.Utils()
bins = 200
eV_to_TeV = 1e-12
TeV_to_erg = 1.60218

known_properties = {
    #PSR J1813-1749 pulsar properties
    'distance': 7e3, #pc
    'longi': 32.638, #deg 
    'lati' : 0.527,  #deg in GCS
    'e_dot' : 9.8e46, #erg/sec
    'char_age' : 43000, # yrs                                                                                                                                  
    'P' :  38.522e-3,  #sec                                                                                                                              
    'P_dot' :  1.4224e-14, #/sec/sec  
    'br_ind' : 3.0, #pulsar braking index  (Assumed)
    'ebreak' : None, #TeV
    'alpha0' : None
}


In [3]:

# model_parameters_assumed_pwn = {
#     #fixed parameters

# }
# model_parameters_fit_pwn = {
#     #fit parameters, lowerbound, upperbound
#     'b_now' : [4.6e-6,1e-6,20e-6], #Gauss
#     'P0' : [22e-3,18e-3,30e-3], #sec
#     'theta' : [0.4469,0.01,1.0], #converstion frac fron pulsar lum to electrons
#     'ecut' : [2.5,1,3], #log10(E/TeV)
#     'alpha' : [2.2,1.8,2.7],
#     'time_frac_xray': [0.8,0.7,0.95], #For x-ray (recent electrons)
#     'time_frac_mediumage': [0.35, 0.2, 0.5],
#     'density' : [1,1,100]#/cm3 #low particle density region
# }
# define_model_parameters_scales(model_parameters_fit_pwn)
# model_parameters_pwn = model_parameters_fit_pwn

# model_parameters_fit_pwn = {
#     #fit parameters, lowerbound, upperbound
#     'b_now' : [4.6e-6,1e-6,20e-6], #Gauss
#     'P0' : [22e-3,18e-3,30e-3], #sec
#     'theta' : [0.4469,0.01,1.0], #converstion frac fron pulsar lum to electrons
#     'ecut' : [2.5,1,3], #log10(E/TeV)
#     'alpha' : [2.2,1.8,2.7],
#     'time_frac_xray': [0.8,0.7,0.95], #For x-ray (recent electrons)
#     'density' : [1,1,100]#/cm3 #low particle density region
# }
# define_model_parameters_scales(model_parameters_fit_pwn)
# model_parameters_pwn = model_parameters_fit_pwn

P0 = 23.7e-03
theta = 0.2
alpha = 1.82
ecut = 3.17
b_now = 2.5e-06
model_parameters_fit_pwn = {
    #fit parameters, lowerbound, upperbound
    'b_now' : [b_now,1e-6,20e-6], #Gauss
    'P0' : [P0,18e-3,40e-3], #sec
    'theta' : [theta,0.01,1.0], #converstion frac fron pulsar lum to electrons
    'ecut' : [ecut,1,3.2], #log10(E/TeV)
    'alpha' : [alpha,1.1,3.0],
    'density' : [1,1,100]#/cm3 #low particle density region
}
# model_parameters_fit_pwn = {
#     #fit parameters, lowerbound, upperbound
#     'b_now' : [4.6e-6,1e-6,20e-6], #Gauss
#     'P0' : [28e-3,18e-3,40e-3], #sec
#     'theta' : [0.4469,0.1,1.0], #converstion frac fron pulsar lum to electrons
#     'ecut' : [2.5,1,3.2], #log10(E/TeV)
#     'alpha' : [2.2,1.1,3.0],
#     'density' : [1,1,100]#/cm3 #low particle density region
# }
define_model_parameters_scales(model_parameters_fit_pwn)
model_parameters_pwn = model_parameters_fit_pwn

In [4]:
for key in model_parameters_pwn.keys():
    print(key,model_parameters_pwn[key])

b_now [2.5e-06, 1e-06, 2e-05, 5263157.894736842]
P0 [0.0237, 0.018, 0.04, 4545.454545454545]
theta [0.2, 0.01, 1.0, 101.01010101010101]
ecut [3.17, 1, 3.2, 45.45454545454545]
alpha [1.82, 1.1, 3.0, 52.631578947368425]
density [1, 1, 100, 1.0101010101010102]


In [5]:
from data import DataDict


hessData = DataDict["HESS"]
fermiData = DataDict["Fermi"]
chandraData = DataDict["Chandra"]
fpmaData = DataDict["NuSTAR"]
km2aData = DataDict["KM2A"]
tibetData = DataDict["Tibet"]
hawcData = DataDict["HAWC"]

/lustre/hawcz01/scratch/userspace/rbabu/astroimage/gamera-time-dependent-scripts


In [6]:
"""""""""""""""""""""""""""
Fitting block : MCMC for PWN

"""""""""""""""""""""""""""
relic_eref = np.concatenate((hawcData['energy'],hessData[hessData['is_ul']==False]['energy'], fermiData[fermiData['is_ul']==False]['energy'], fpmaData[fpmaData['is_ul']==False]['energy'], km2aData['energy'], tibetData['energy'], chandraData['energy']))
relic_sed = np.concatenate((hawcData['flux'],hessData[hessData['is_ul']==False]['flux'], fermiData[fermiData['is_ul']==False]['flux'], fpmaData[fpmaData['is_ul']==False]['flux'], km2aData['flux'], tibetData['flux'], chandraData['flux']))
relic_sed_err_lo = np.concatenate((hawcData['flux_error_lo'],hessData[hessData['is_ul']==False]['flux_error_lo'], fermiData[fermiData['is_ul']==False]['flux_error_lo'], fpmaData[fpmaData['is_ul']==False]['flux_error_lo'], km2aData['flux_error_lo'], tibetData['flux_error_lo'], chandraData['flux_error_lo']))
relic_sed_err_hi = np.concatenate((hawcData['flux_error_hi'],hessData[hessData['is_ul']==False]['flux_error_hi'], fermiData[fermiData['is_ul']==False]['flux_error_hi'], fpmaData[fpmaData['is_ul']==False]['flux_error_hi'], km2aData['flux_error_hi'], tibetData['flux_error_hi'], chandraData['flux_error_hi']))



# relic_eref = np.concatenate((HessCompAData[HessCompAData['ul']==False]['energy'], HAWCCompAData['energy'], HessCompBData[HessCompBData['ul']==False]['energy'], HAWCCompBData['energy'], integralData['energy'], chandraData['energy']))
# relic_sed = np.concatenate((HessCompAData[HessCompAData['ul']==False]['flux'], HAWCCompAData['flux'], HessCompBData[HessCompBData['ul']==False]['flux'], HAWCCompBData['flux'], integralData['flux'], chandraData['flux']))
# relic_sed_err_lo = np.concatenate((HessCompAData[HessCompAData['ul']==False]['flux_error_lo'], HAWCCompAData['flux_error_lo'], HessCompBData[HessCompBData['ul']==False]['flux_error_lo'], HAWCCompBData['flux_error_lo'], integralData['flux_error_lo'], chandraData['flux_error_lo']))
# relic_sed_err_hi = np.concatenate((HessCompAData[HessCompAData['ul']==False]['flux_error_hi'], HAWCCompAData['flux_error_hi'], HessCompBData[HessCompBData['ul']==False]['flux_error_hi'], HAWCCompBData['flux_error_hi'], integralData['flux_error_hi'], chandraData['flux_error_hi']))

# middle_age_pwn_eref = np.concatenate((HessCompBData[HessCompBData['ul']==False]['energy'], HAWCCompBData['energy'], integralData['energy'], chandraData['energy']))
# middle_age_pwn_sed = np.concatenate((HessCompBData[HessCompBData['ul']==False]['flux'], HAWCCompBData['flux'], integralData['flux'], chandraData['flux']))
# middle_age_pwn_err_lo =  np.concatenate((HessCompBData[HessCompBData['ul']==False]['flux_error_lo'], HAWCCompBData['flux_error_lo'], integralData['flux_error_lo'], chandraData['flux_error_lo']))
# middle_age_pwn_err_hi = np.concatenate((HessCompBData[HessCompBData['ul']==False]['flux_error_hi'], HAWCCompBData['flux_error_hi'], integralData['flux_error_hi'], chandraData['flux_error_hi']))

# young_age_eref =  np.concatenate((integralData['energy'], chandraData['energy']))
# young_age_sed =  np.concatenate((integralData['flux'], chandraData['flux']))
# young_age_err_lo =  np.concatenate((integralData['flux_error_lo'], chandraData['flux_error_lo']))
# young_age_err_hi =  np.concatenate((integralData['flux_error_hi'], chandraData['flux_error_hi']))

# pwn_x = [young_age_eref, middle_age_pwn_eref, relic_eref]
# pwn_y = [young_age_sed, middle_age_pwn_sed, relic_sed]
# pwn_yerr_lo = [young_age_err_lo, middle_age_pwn_err_lo, relic_sed_err_lo]
# pwn_yerr_hi = [young_age_err_hi, middle_age_pwn_err_hi, relic_sed_err_hi]

# pwn_x = [middle_age_pwn_eref, relic_eref]
# pwn_y = [middle_age_pwn_sed, relic_sed]
# pwn_yerr_lo = [middle_age_pwn_err_lo, relic_sed_err_lo]
# pwn_yerr_hi = [middle_age_pwn_err_hi, relic_sed_err_hi]

pwn_x = relic_eref
pwn_y = relic_sed
pwn_yerr_lo = relic_sed_err_lo
pwn_yerr_hi = relic_sed_err_hi

# pwn_x = np.array((np.array(integralData['energy']), middle_age_pwn_eref, relic_eref))*gp.TeV_to_erg #should be in ergs for gamera
# pwn_y = np.array((np.array(integralData['flux']), middle_age_pwn_sed, relic_sed))
# pwn_yerr = np.array((np.array(integralData['flux_error_lo']), middle_age_pwn_err_lo,  young_age_err_lo))

# pwn_x = np.array(np.array(young_age_eref, middle_age_pwn_eref, relic_eref))
# pwn_y = np.array((np.array[young_age_sed, middle_age_pwn_sed, relic_sed]))
# pwn_yerr_lo = np.array((np.array[young_age_err_lo, middle_age_pwn_err_lo, relic_sed_err_lo]))
# pwn_yerr_hi = np.array((np.array[young_age_err_hi, middle_age_pwn_err_hi, relic_sed_err_hi]))

print("Relic size", relic_eref.size, relic_sed.size, relic_sed_err_lo.size)
# print("Middle size", middle_age_pwn_eref.size, middle_age_pwn_sed.size, middle_age_pwn_err_lo.size)
# print("PWN size", pwn_x.size, pwn_y.size, pwn_yerr_lo.size)

Relic size 39 39 39


In [7]:
# pwn_fit_parameters = ['theta','b_now','P0','ecut','alpha','time_frac_xray', 'time_frac_mediumage']
# pwn_labels = ["Theta (fraction)", "B(Now)(G)", "P0 (s)", "log10(Ecut/TeV)", "spectral index (wind)","Time fraction (X-ray)", "Time fraction (middle-aged)"]
# pwn_fit = pwn_emission_threezone(bins, known_properties, model_parameters_pwn)
# pwn_pars = []
# pwn_lower_bounds = []
# pwn_upper_bounds = []
# for par in pwn_fit_parameters:
#     pwn_pars.append(model_parameters_pwn[par][0]*model_parameters_pwn[par][-1])
#     pwn_lower_bounds.append(model_parameters_pwn[par][1]*model_parameters_pwn[par][-1])
#     pwn_upper_bounds.append(model_parameters_pwn[par][2]*model_parameters_pwn[par][-1])
# pwn_bounds = (pwn_lower_bounds, pwn_upper_bounds)
# def proxy(cls_instance):
#     return cls_instance.log_prob_pwn
# mcmc_result_pwn = run_mcmc(7,pwn_pars,pwn_fit.log_prob_pwn,pwn_x,pwn_y,pwn_yerr_lo)

In [8]:
# pwn_fit_parameters = ['theta','b_now','P0','ecut','alpha','time_frac_xray']
# pwn_labels = ["Theta (fraction)", "B(Now)(G)", "P0 (s)", "log10(Ecut/TeV)", "spectral index (wind)","Time fraction (X-ray)",]
# pwn_fit = pwn_emission_twozone(bins, known_properties, model_parameters_pwn)
# pwn_pars = []
# pwn_lower_bounds = []
# pwn_upper_bounds = []
# for par in pwn_fit_parameters:
#     pwn_pars.append(model_parameters_pwn[par][0]*model_parameters_pwn[par][-1])
#     pwn_lower_bounds.append(model_parameters_pwn[par][1]*model_parameters_pwn[par][-1])
#     pwn_upper_bounds.append(model_parameters_pwn[par][2]*model_parameters_pwn[par][-1])
# pwn_bounds = (pwn_lower_bounds, pwn_upper_bounds)
# def proxy(cls_instance):
#     return cls_instance.log_prob_pwn
# mcmc_result_pwn = run_mcmc(6,pwn_pars,pwn_fit.log_prob_pwn,pwn_x,pwn_y,pwn_yerr_lo)

In [9]:
pwn_fit_parameters = ['theta','b_now','P0','ecut','alpha',]
pwn_labels = ["Theta (fraction)", "B(Now)(G)", "P0 (s)", "log10(Ecut/TeV)", "spectral index (wind)"]
pwn_fit = pwn_emission_singlezone(bins, known_properties, model_parameters_pwn)
pwn_pars = []
pwn_lower_bounds = []
pwn_upper_bounds = []
for par in pwn_fit_parameters:
    pwn_pars.append(model_parameters_pwn[par][0]*model_parameters_pwn[par][-1])
    pwn_lower_bounds.append(model_parameters_pwn[par][1]*model_parameters_pwn[par][-1])
    pwn_upper_bounds.append(model_parameters_pwn[par][2]*model_parameters_pwn[par][-1])
pwn_bounds = (pwn_lower_bounds, pwn_upper_bounds)
def proxy(cls_instance):
    return cls_instance.log_prob_pwn


Initialized model parameters: b_now 2.5e-06 P0 0.0237 theta 0.2 ecut 3.17 alpha 1.82 density 1


In [10]:
pwn_fit = pwn_emission_singlezone(bins, known_properties, model_parameters_pwn)

print(f"self.P0       = {pwn_fit.P0:.3e} s    (should be ~30e-3)")
print(f"self.P        = {pwn_fit.P:.3e} s    (should be ~38.5e-3)")
print(f"self.P_dot    = {pwn_fit.P_dot:.3e}   (should be ~1.4e-14)")
print(f"self.br_ind   = {pwn_fit.br_ind}")
print(f"self.theta    = {pwn_fit.theta:.3e}")
print(f"self.e_dot    = {pwn_fit.e_dot:.3e} erg/s")
print()
print(f"t0            = {pwn_fit.calculate_t0():.3e} yr")
print(f"true_age      = {pwn_fit.calculate_true_age():.3e} yr")
print(f"br_ind_factor = {pwn_fit.calculate_br_ind_power_factor():.3f}")
print(f"L0            = {pwn_fit.calculate_l0():.3e} erg/s")
print(f"B0            = {pwn_fit.calculate_b0():.3e} G")
print()
# Check luminosity at a few times
for t in [1, 100, 1000, 10000, 43000]:
    print(f"L(t={t:6d} yr) = {pwn_fit.luminosity(t):.3e} erg/s")


Initialized model parameters: b_now 2.5e-06 P0 0.0237 theta 0.2 ecut 3.17 alpha 1.82 density 1
self.P0       = 2.370e-02 s    (should be ~30e-3)
self.P        = 3.852e-02 s    (should be ~38.5e-3)
self.P_dot    = 1.422e-14   (should be ~1.4e-14)
self.br_ind   = 3.0
self.theta    = 2.000e-01
self.e_dot    = 9.800e+46 erg/s

t0            = 1.624e+04 yr
true_age      = 2.667e+04 yr
br_ind_factor = 2.000
L0            = 6.840e+47 erg/s
B0            = 5.703e-06 G

L(t=     1 yr) = 1.368e+47 erg/s
L(t=   100 yr) = 1.351e+47 erg/s
L(t=  1000 yr) = 1.214e+47 erg/s
L(t= 10000 yr) = 5.241e+46 erg/s
L(t= 43000 yr) = 1.028e+46 erg/s


In [11]:
print(f"true_age = {pwn_fit.calculate_true_age():.3e} yr")
print(f"twindow_sec = {pwn_fit.calculate_true_age() * yr_to_sec:.3e} s")
# Should be ~26700 yr * 3.156e7 s/yr = ~8.4e11 s

true_age = 2.667e+04 yr
twindow_sec = 8.416e+11 s


In [12]:
t_yr = np.logspace(0, 5, 10)
e_electron = np.logspace(np.log10(gp.m_e/gp.TeV_to_erg), 4, 50) * gp.TeV_to_erg
e_photon = np.logspace(-6, 15, 50) * gp.eV_to_erg


pwn_fit.theta  = model_parameters_pwn['theta'][0]
pwn_fit.b_now  = model_parameters_pwn['b_now'][0]
pwn_fit.P0     = model_parameters_pwn['P0'][0]
pwn_fit.ecut   = model_parameters_pwn['ecut'][0]
pwn_fit.alpha  = model_parameters_pwn['alpha'][0]


In [ ]:
pwn_sed_test, fp_test, fr_test = pwn_fit.model_pwn(t_yr, e_electron, e_photon, 1, False)

In [ ]:
# t_yr = np.logspace(0, 5, 200)



sp = np.array(fp_test.GetParticleSpectrum())
total_e = np.trapz(sp[:,1]*sp[:,0], sp[:,0])
print(f"Total electron energy: {total_e:.3e} erg  (should be ~1.6e58)")
print(f"Model SED flux range:  {pwn_sed_test[:,1].min():.3e} to {pwn_sed_test[:,1].max():.3e} erg/cm2/s")
print(f"Data flux range:       {pwn_y.min():.3e} to {pwn_y.max():.3e} erg/cm2/s")

In [ ]:
import matplotlib.pyplot as plt

# Get the SED with initial parameters
pwn_sed_relic, fp_pwn, fr_pwn = pwn_fit.get_pwn_sed(
    [model_parameters_pwn[p][0] * model_parameters_pwn[p][-1] 
     for p in pwn_fit_parameters]
)

print("Model SED energy range:", pwn_sed_relic[:,0].min(), "to", pwn_sed_relic[:,0].max(), "erg")
print("Model SED flux range:  ", pwn_sed_relic[:,1].min(), "to", pwn_sed_relic[:,1].max(), "erg/cm2/s")
print()
print("Data energy range:", pwn_x.min(), "to", pwn_x.max(), "erg")
print("Data flux range:  ", pwn_y.min(), "to", pwn_y.max(), "erg/cm2/s")

# Plot both on same axes to see the offset
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(pwn_sed_relic[:,0], pwn_sed_relic[:,1], label='Model')
ax.errorbar(pwn_x, pwn_y, yerr=[pwn_yerr_lo, pwn_yerr_hi], 
            fmt='o', label='Data')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Energy (erg)')
ax.set_ylabel('E²dN/dE (erg/cm²/s)')
ax.legend()
plt.savefig("plots/initial_model_vs_data.png", bbox_inches='tight')
print("Saved to plots/initial_model_vs_data.png")

In [ ]:
sp = np.array(fp_pwn.GetParticleSpectrum())
print(f"\nParticle spectrum: {sp.shape}")
print(f"Electron energy range: {sp[:,0].min():.3e} to {sp[:,0].max():.3e} erg")
print(f"Electron dN/dE range:  {sp[:,1].min():.3e} to {sp[:,1].max():.3e}")

# Check total electron energy
total_e = np.trapz(sp[:,1]*sp[:,0], sp[:,0])
print(f"Total electron energy: {total_e:.3e} erg")
print(f"Expected (~theta*Edot*age): {pwn_fit.theta * pwn_fit.e_dot * pwn_fit.calculate_true_age() * gp.yr_to_sec:.3e} erg")
print(f"\nDistance used: {pwn_fit.distance} pc")
print(f"In cm: {pwn_fit.distance * 3.086e18:.3e} cm")

In [ ]:
# Check what injection spectrum looks like before MCMC
test_e = np.logspace(np.log10(gp.m_e/gp.TeV_to_erg), 4, 200) * gp.TeV_to_erg
test_spec = pwn_fit.injection_spectrum_pwn(test_e)
print("Injection spectrum range:", np.array(test_spec)[:,1].min(), np.array(test_spec)[:,1].max())

# Check data energy units
print("Energy range of data:", relic_eref.min(), relic_eref.max())
print("Flux range of data:", relic_sed.min(), relic_sed.max())

In [ ]:
for name, data in DataDict.items():
    e = data['energy']
    print(f"{name}: E range = {e.min():.3e} to {e.max():.3e} erg "
          f"= {e.min()/1.60218e-12:.3e} to {e.max()/1.60218e-12:.3e} TeV")

In [ ]:
import pandas as pd
for name, path in [("Chandra","data/chandraData.csv"), 
                   ("HESS","data/hessData.csv"),
                   ("Fermi","data/fermiData.csv"),
                   ("KM2A","data/km2aData.csv"),
                   ("Tibet","data/tibetData.csv"),
                   ("NuSTAR","data/fpmaData.csv"),
                   ("HAWC", "data/hawcData.csv")]:
    df = pd.read_csv(path)
    print(f"{name}: E = {df['energy'].min():.3e} to {df['energy'].max():.3e} erg, "
          f"flux = {df['flux'].min():.3e} to {df['flux'].max():.3e} erg/cm2/s")


In [ ]:
test_e = np.logspace(np.log10(gp.m_e/gp.TeV_to_erg), 4, 10) * gp.TeV_to_erg
for ei in test_e:
    ecut = np.power(10, 2.5) * gp.TeV_to_erg
    e0   = 1.0 * gp.TeV_to_erg
    pwl  = (ei / e0) ** -2.2
    spec = pwl * np.exp(-ei / ecut)
    print(f"e={ei:.3e} erg  ({ei/gp.TeV_to_erg:.3e} TeV)  pwl={pwl:.3e}  spec={spec:.3e}")

In [ ]:
print(type(pwn_fit))
print(pwn_fit.theta_scale)
print(pwn_fit.b_now_scale)

In [ ]:
# P0 = 23.7e-03
# theta = 0.95
# alpha = 2
# ecut = 1.6
# b_now = 7.5e-06
P0 = 23.7e-03
theta = 0.015
alpha = 1.82
ecut = 2.4
b_now = 2.5e-06
paper_results = [theta * pwn_fit.theta_scale,
                 b_now * pwn_fit.b_now_scale,
                 P0    * pwn_fit.P0_scale,
                 ecut  * pwn_fit.ecut_scale,
                 alpha * pwn_fit.alpha_scale]
print(paper_results)
print("theta:", theta * pwn_fit.theta_scale)
print("b_now:", b_now * pwn_fit.b_now_scale)
print("P0:", P0 * pwn_fit.P0_scale)
print("ecut:", ecut * pwn_fit.ecut_scale)
print("alpha:", alpha * pwn_fit.alpha_scale)
print("paper_results:", paper_results)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gappa as gp

# same electron grid used in get_pwn_sed
bins = 200
e_electron = np.logspace(np.log10(gp.m_e / gp.TeV_to_erg), 4, bins) * gp.TeV_to_erg

# e_sed is list of (E[erg], dN/dE)
e_sed = pwn_fit.injection_spectrum_pwn(e_electron)
e_sed_arr = np.array(e_sed, dtype=float)

# plot in TeV for x-axis
E_TeV = e_sed_arr[:, 0] / gp.TeV_to_erg
dNdE = e_sed_arr[:, 1]

plt.figure(figsize=(7, 5))
plt.loglog(E_TeV, dNdE)
plt.xlabel("Electron energy [TeV]")
plt.ylabel("Injection spectrum dN/dE [1/erg]")
# plt.ylim(1e-20, 1e-10)
plt.title("Injected electron spectrum (e_sed)")
plt.grid(True, which="both", ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
import inspect
print(inspect.signature(pwn_fit.get_pwn_sed))
print(pwn_fit_parameters)
print("paper_results:", paper_results)
print(inspect.getsource(pwn_fit.get_pwn_sed))

In [ ]:
paper_results = [theta * pwn_fit.theta_scale,
                 b_now * pwn_fit.b_now_scale,
                 P0    * pwn_fit.P0_scale,
                 ecut  * pwn_fit.ecut_scale,
                 alpha * pwn_fit.alpha_scale]

In [ ]:
284018400000/gp.yr_to_sec

In [ ]:
# pwn_sed_relic, fp_pwn, fr_pwn = pwn_fit.get_pwn_sed(paper_results)
pwn_sed_relic, fp_pwn, fr_pwn = pwn_fit.get_pwn_sed(paper_results)

In [ ]:
def draw_observations_data2(zoom=False):
    fig, ax = plt.subplots(figsize=(14, 9))
    
    # --- HESS ---
    hess_det = hessData[hessData['is_ul']==False]
    hess_ul  = hessData[hessData['is_ul']==True]
    ax.errorbar(hess_det['energy'], hess_det['flux'],
                yerr=[hess_det['flux_error_lo'], hess_det['flux_error_hi']],
                marker='o', linestyle='None', label='HESS',
                mfc='fuchsia', mec='fuchsia', ecolor='fuchsia', capsize=3, fmt='o', zorder=5)
    if len(hess_ul) > 0:
        ax.errorbar(hess_ul['energy'], hess_ul['flux'],
                    yerr=0.3*hess_ul['flux'], uplims=True,
                    marker='v', linestyle='None',
                    mfc='fuchsia', mec='fuchsia', ecolor='fuchsia', capsize=3, fmt='v', zorder=5)

    # --- Fermi ---
    fermi_det = fermiData[fermiData['is_ul']==False]
    fermi_ul  = fermiData[fermiData['is_ul']==True]
    ax.errorbar(fermi_det['energy'], fermi_det['flux'],
                yerr=[fermi_det['flux_error_lo'], fermi_det['flux_error_hi']],
                marker='s', linestyle='None', label='Fermi',
                mfc='lime', mec='lime', ecolor='lime', capsize=3, fmt='s', zorder=5)
    if len(fermi_ul) > 0:
        ax.errorbar(fermi_ul['energy'], fermi_ul['flux'],
                    yerr=0.3*fermi_ul['flux'], uplims=True,
                    marker='v', linestyle='None',
                    mfc='lime', mec='lime', ecolor='lime', capsize=3, fmt='v', zorder=5)

    # --- FPMA ---
    fpma_det = fpmaData[fpmaData['is_ul']==False]
    fpma_ul  = fpmaData[fpmaData['is_ul']==True]
    ax.errorbar(fpma_det['energy'], fpma_det['flux'],
                yerr=[fpma_det['flux_error_lo'], fpma_det['flux_error_hi']],
                marker='^', linestyle='None', label='FPMA',
                mfc='violet', mec='violet', ecolor='violet', capsize=3, fmt='^', zorder=5)
    if len(fpma_ul) > 0:
        ax.errorbar(fpma_ul['energy'], fpma_ul['flux'],
                    yerr=0.3*fpma_ul['flux'], uplims=True,
                    marker='v', linestyle='None',
                    mfc='violet', mec='violet', ecolor='violet', capsize=3, fmt='v', zorder=5)

    # --- KM2A ---
    ax.errorbar(km2aData['energy'], km2aData['flux'],
                yerr=[km2aData['flux_error_lo'], km2aData['flux_error_hi']],
                marker='D', linestyle='None', label='KM2A',
                mfc='red', mec='red', ecolor='red', capsize=3, fmt='D', zorder=5)

    # --- Tibet ---
    ax.errorbar(tibetData['energy'], tibetData['flux'],
                yerr=[tibetData['flux_error_lo'], tibetData['flux_error_hi']],
                marker='P', linestyle='None', label='Tibet',
                mfc='orange', mec='orange', ecolor='orange', capsize=3, fmt='P', zorder=5)

    # --- Chandra ---
    ax.errorbar(chandraData['energy'], chandraData['flux'],
                yerr=[chandraData['flux_error_lo'], chandraData['flux_error_hi']],
                marker='*', linestyle='None', label='Chandra',
                mfc='deepskyblue', mec='deepskyblue', ecolor='deepskyblue', capsize=3, fmt='*', zorder=5, markersize=10)

    # --- Model ---
    ax.plot(pwn_sed_relic[:,0], pwn_sed_relic[:,1]*1e-14,
            linestyle='-', color='steelblue', linewidth=2, label='Total model', zorder=4)

    ax.set_xscale('log')
    ax.set_yscale('log')

    ax.set_title(r'HESS J1813 $\gamma$-ray Flux Points', size=20, pad=12)
    ax.set_ylabel(r'E$_{\gamma}^{2}\Phi_{\gamma}$ [erg/(cm${}^{2}$ s)]', size=16)
    ax.set_xlabel(r'E$_{\gamma}$ (TeV)', size=16)

    if zoom:
        ax.set_xlim(0.8e-3, 1e3)
        ax.set_ylim(bottom=8e-15)
    else:
        ax.set_xlim(5e-19, 1e3)
        ax.set_ylim(bottom=8e-15)

    leg = ax.legend(loc='upper left', fontsize=13, ncol=2, framealpha=0.9,
                    fancybox=False, edgecolor='grey')
    leg.get_frame().set_linewidth(0.8)

    ax.grid(which='both', axis='both', linestyle='--', linewidth=0.4, alpha=0.6)
    ax.tick_params(axis='both', labelsize=13, which='both', direction='in',
                   top=True, right=True, length=5)
    ax.tick_params(axis='both', which='minor', length=3)

    fig.tight_layout()
    plt.show()
draw_observations_data2(zoom=False)

In [ ]:
import os
usable_threads = len(os.sched_getaffinity(0))
print("Usable threads for this notebook:", usable_threads)
print("Total logical CPUs on node:", os.cpu_count())

In [ ]:
mcmc_result_pwn = run_mcmc(5,pwn_pars,pwn_fit.log_prob_pwn,pwn_x,pwn_y,pwn_yerr_lo, num_threads=32,num_walkers=10, num_burn_in=100, chain_steps=300)

In [ ]:
with open('mcmc_results/hawc_mcmc_result_pwn_notebook.npy', 'wb') as f:
    np.save(f, mcmc_result_pwn)

In [ ]:
mcmc_result_pwn = np.load('mcmc_results/hawc_mcmc_result_pwn_notebook.npy', allow_pickle=True)

In [ ]:
mcmc_result_pwn 

In [ ]:
mcmc_pwn_fit_results, mcmc_pwn_fit_results_err_high, mcmc_pwn_fit_results_err_low = mcmc_results("pwn", pwn_labels, pwn_fit_parameters, model_parameters_pwn, mcmc_result_pwn)
plt.show()
# plt.savefig("plots/pwn_cornerplot.png", bbox_inches='tight')

In [ ]:
# obj_fit=pwn_fit
# fit_values = mcmc_pwn_fit_results

# theta = fit_values[0]/obj_fit.theta_scale
# b_now = fit_values[1]/obj_fit.b_now_scale
# P0 = fit_values[2]/obj_fit.P0_scale
# ecut = fit_values[3]/obj_fit.ecut_scale
# alpha = fit_values[4]/obj_fit.alpha_scale
# print(f"Theta {theta}, B(Now) {b_now}, P0 {P0}, Ecut {ecut}, alpha {alpha}")

In [ ]:
# P0 = 23.7e-03
# theta = 0.012
# alpha = 1.82
# ecut = 3.176
# b_now = 2.5e-06

# # paper_results = [P0*obj_fit.P0_scale, b_now*obj_fit.b_now_scale, theta*obj_fit.theta_scale, ecut*obj_fit.ecut_scale, alpha*obj_fit.alpha_scale]

# paper_results = [theta*obj_fit.theta_scale, b_now*obj_fit.b_now_scale, P0*obj_fit.P0_scale, ecut*obj_fit.ecut_scale, alpha*obj_fit.alpha_scale]

# print(mcmc_pwn_fit_results)
# print(paper_results)

In [ ]:
# pwn_fit.calculate_true_age()*gp.yr_to_sec

In [ ]:
# mcmc_pwn_fit_results

In [ ]:
# P0 = 23.7e-03
# theta = 0.95
# alpha = 2
# ecut = 1.6
# b_now = 7.5e-06

# # paper_results = [P0*obj_fit.P0_scale, b_now*obj_fit.b_now_scale, theta*obj_fit.theta_scale, ecut*obj_fit.ecut_scale, alpha*obj_fit.alpha_scale]
# obj_fit=pwn_fit
# paper_results = [theta*obj_fit.theta_scale, b_now*obj_fit.b_now_scale, P0*obj_fit.P0_scale, ecut*obj_fit.ecut_scale, alpha*obj_fit.alpha_scale]

# # print(mcmc_pwn_fit_results)
# print(paper_results)
# print("theta:", theta * pwn_fit.theta_scale)
# print("b_now:", b_now * pwn_fit.b_now_scale)
# print("P0:", P0 * pwn_fit.P0_scale)
# print("ecut:", ecut * pwn_fit.ecut_scale)
# print("alpha:", alpha * pwn_fit.alpha_scale)
# print("paper_results:", paper_results)
# pwn_sed_relic, fp_pwn, fr_pwn = pwn_emission_singlezone.get_pwn_sed(pwn_fit, paper_results)
# # pwn_sed_relic, fp_pwn, fr_pwn = get_pwn_sed(pwn_fit, mcmc_pwn_fit_results)

In [ ]:
# print("theta:", theta * pwn_fit.theta_scale)
# print("b_now:", b_now * pwn_fit.b_now_scale)
# print("P0:", P0 * pwn_fit.P0_scale)
# print("ecut:", ecut * pwn_fit.ecut_scale)
# print("alpha:", alpha * pwn_fit.alpha_scale)
# print("paper_results:", paper_results)

In [ ]:
# pwn_sed_relic, fp_pwn, fr_pwn = pwn_fit.get_pwn_sed(paper_results)
pwn_sed_relic, fp_pwn, fr_pwn = pwn_fit.get_pwn_sed(mcmc_pwn_fit_results)

In [ ]:
def draw_observations_data2(zoom=False):
    fig, ax = plt.subplots(figsize=(14, 9))
    
    # --- HESS ---
    hess_det = hessData[hessData['is_ul']==False]
    hess_ul  = hessData[hessData['is_ul']==True]
    ax.errorbar(hess_det['energy'], hess_det['flux'],
                yerr=[hess_det['flux_error_lo'], hess_det['flux_error_hi']],
                marker='o', linestyle='None', label='HESS',
                mfc='fuchsia', mec='fuchsia', ecolor='fuchsia', capsize=3, fmt='o', zorder=5)
    if len(hess_ul) > 0:
        ax.errorbar(hess_ul['energy'], hess_ul['flux'],
                    yerr=0.3*hess_ul['flux'], uplims=True,
                    marker='v', linestyle='None',
                    mfc='fuchsia', mec='fuchsia', ecolor='fuchsia', capsize=3, fmt='v', zorder=5)

    # --- Fermi ---
    fermi_det = fermiData[fermiData['is_ul']==False]
    fermi_ul  = fermiData[fermiData['is_ul']==True]
    ax.errorbar(fermi_det['energy'], fermi_det['flux'],
                yerr=[fermi_det['flux_error_lo'], fermi_det['flux_error_hi']],
                marker='s', linestyle='None', label='Fermi',
                mfc='lime', mec='lime', ecolor='lime', capsize=3, fmt='s', zorder=5)
    if len(fermi_ul) > 0:
        ax.errorbar(fermi_ul['energy'], fermi_ul['flux'],
                    yerr=0.3*fermi_ul['flux'], uplims=True,
                    marker='v', linestyle='None',
                    mfc='lime', mec='lime', ecolor='lime', capsize=3, fmt='v', zorder=5)

    # --- FPMA ---
    fpma_det = fpmaData[fpmaData['is_ul']==False]
    fpma_ul  = fpmaData[fpmaData['is_ul']==True]
    ax.errorbar(fpma_det['energy'], fpma_det['flux'],
                yerr=[fpma_det['flux_error_lo'], fpma_det['flux_error_hi']],
                marker='^', linestyle='None', label='FPMA',
                mfc='violet', mec='violet', ecolor='violet', capsize=3, fmt='^', zorder=5)
    if len(fpma_ul) > 0:
        ax.errorbar(fpma_ul['energy'], fpma_ul['flux'],
                    yerr=0.3*fpma_ul['flux'], uplims=True,
                    marker='v', linestyle='None',
                    mfc='violet', mec='violet', ecolor='violet', capsize=3, fmt='v', zorder=5)

    # --- KM2A ---
    ax.errorbar(km2aData['energy'], km2aData['flux'],
                yerr=[km2aData['flux_error_lo'], km2aData['flux_error_hi']],
                marker='D', linestyle='None', label='KM2A',
                mfc='red', mec='red', ecolor='red', capsize=3, fmt='D', zorder=5)

    # --- Tibet ---
    ax.errorbar(tibetData['energy'], tibetData['flux'],
                yerr=[tibetData['flux_error_lo'], tibetData['flux_error_hi']],
                marker='P', linestyle='None', label='Tibet',
                mfc='orange', mec='orange', ecolor='orange', capsize=3, fmt='P', zorder=5)

    # --- Chandra ---
    ax.errorbar(chandraData['energy'], chandraData['flux'],
                yerr=[chandraData['flux_error_lo'], chandraData['flux_error_hi']],
                marker='*', linestyle='None', label='Chandra',
                mfc='deepskyblue', mec='deepskyblue', ecolor='deepskyblue', capsize=3, fmt='*', zorder=5, markersize=10)

    # --- Model ---
    ax.plot(pwn_sed_relic[:,0], pwn_sed_relic[:,1],
            linestyle='-', color='steelblue', linewidth=2, label='Total model', zorder=4)

    ax.set_xscale('log')
    ax.set_yscale('log')

    ax.set_title(r'HESS J1813 $\gamma$-ray Flux Points', size=20, pad=12)
    ax.set_ylabel(r'E$_{\gamma}^{2}\Phi_{\gamma}$ [erg/(cm${}^{2}$ s)]', size=16)
    ax.set_xlabel(r'E$_{\gamma}$ (TeV)', size=16)

    if zoom:
        ax.set_xlim(0.8e-3, 1e3)
        ax.set_ylim(bottom=8e-15)
    else:
        ax.set_xlim(5e-19, 1e3)
        ax.set_ylim(bottom=8e-15)

    leg = ax.legend(loc='upper left', fontsize=13, ncol=2, framealpha=0.9,
                    fancybox=False, edgecolor='grey')
    leg.get_frame().set_linewidth(0.8)

    ax.grid(which='both', axis='both', linestyle='--', linewidth=0.4, alpha=0.6)
    ax.tick_params(axis='both', labelsize=13, which='both', direction='in',
                   top=True, right=True, length=5)
    ax.tick_params(axis='both', which='minor', length=3)

    fig.tight_layout()
    plt.show()
draw_observations_data2(zoom=False)